# Train the vessel detector (YOLOv8s) on Google Colab

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Run the cells from top to bottom (▶ on each cell). Total time ≈ 2–3 hours, mostly step 5.
Keep this browser tab open while it trains; Colab disconnects idle sessions.

| Step | What it does | Time |
|---|---|---|
| 1 | Get the project from GitHub, install packages | 2–3 min |
| 2 | Connect Google Drive (checkpoints are saved there) | 1 min |
| 3 | Download the dataset from Roboflow (asks for your API key) | 2–5 min |
| 4 | Split by video + one "vessel" class + night/haze copies | 3–5 min |
| 5 | **Train YOLOv8s** (100 epochs, stops early if no improvement for 30) | 1.5–2.5 h |
| 6 | Test accuracy on unseen videos: overall, visible, NIR, haze + FPS | 5 min |
| 7 | Download the trained model and results to your PC | 1 min |

In [ ]:
# 1) Project + packages
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /content/project && git clone -q https://github.com/naman-c25/Maritime-Anomaly-Detection.git /content/project
%cd /content/project
!pip -q install -r computer_vision/requirements-train.txt

In [ ]:
# 2) Google Drive: training checkpoints are written here, so a disconnect does not lose the run
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/maritime_runs'
!mkdir -p {RUNS}

In [ ]:
# 3) Dataset from Roboflow (Roboflow → Settings → API Keys → copy the private key)
from getpass import getpass
from roboflow import Roboflow
key = getpass('Roboflow API key: ')
project = Roboflow(api_key=key).workspace('naman-kumar-chaudhary').project('singapore-maritime-bvyeu')
project.version(1).download('yolov8', location='/content/rf_export', overwrite=True)

In [ ]:
# 4) Re-split by video (no recording in two splits) and convert to a single 'vessel' class
!rm -rf computer_vision/data/maritime
!python computer_vision/scripts/resplit_by_video.py --src /content/rf_export --dst computer_vision/data/maritime
!python computer_vision/scripts/prepare_data.py --root computer_vision/data/maritime --conditions-csv computer_vision/data/maritime/conditions.csv

### Step 5: training
While it runs, each epoch prints `box_loss` (should go down) and `mAP50` on the validation videos (should go up).

**If Colab disconnects:** run cells 1, 2 and 4 again (skip 3 only if `/content/rf_export` still exists), then run
the *resume* cell below instead of the training cell; it continues from the last saved epoch on Drive.

In [ ]:
# 5) Train YOLOv8s
!python computer_vision/scripts/train.py --data computer_vision/data/maritime/data_vessel.yaml --weights yolov8s.pt --device 0 --project {RUNS}

In [ ]:
# 5b) ONLY after a disconnect: continue the interrupted run
!python computer_vision/scripts/train.py --resume {RUNS}/vessel_yolov8s/weights/last.pt

In [ ]:
# 6) Test accuracy on the 10 unseen test videos (overall + visible / nir / haze) and speed on the T4
!python computer_vision/scripts/evaluate_detector.py --weights computer_vision/models/vessel_yolov8s_best.pt --data computer_vision/data/maritime/data_vessel.yaml --device 0

In [ ]:
# 7) Download: trained model + test results + training curves
!mkdir -p export/computer_vision/models export/computer_vision/results export/training_run
!cp computer_vision/models/vessel_yolov8s_best.pt export/computer_vision/models/
!cp computer_vision/results/*.json export/computer_vision/results/
!cp {RUNS}/vessel_yolov8s/*.png {RUNS}/vessel_yolov8s/results.csv export/training_run/ 2>/dev/null; true
!cd export && zip -q -r ../trained_vessel_model.zip . && cd ..
!cp trained_vessel_model.zip {RUNS}/
from google.colab import files
files.download('trained_vessel_model.zip')

## After downloading (on your PC)
1. Extract `trained_vessel_model.zip` into `F:\AI_ML` and replace files if asked. This puts
   `computer_vision\models\vessel_yolov8s_best.pt` and `computer_vision\results\detector_vessel_yolov8s_best.json`
   in place. `training_run\` holds the curves (results.png, confusion matrix) for the report.
2. In `F:\AI_ML` run:
   ```
   git add computer_vision/models/vessel_yolov8s_best.pt computer_vision/results
   git commit -m "Trained vessel detector"
   git push
   ```
3. The website redeploys by itself and uses the new model; the Analytics page shows the test results.

## Optional: baselines for the paper (Faster R-CNN / SSD, several hours each)

In [ ]:
# Optional baselines (Table 8 of the paper)
!python computer_vision/scripts/train_baselines.py --data computer_vision/data/maritime/data_vessel.yaml --model ssd --epochs 60
!python computer_vision/scripts/train_baselines.py --data computer_vision/data/maritime/data_vessel.yaml --model fasterrcnn --epochs 30